# Time-to-event model comparison with CTG-only FRI Cox baseline

This copy of `postprocess_time_to_event_models.ipynb` combines the completed deep-learning landmark artifacts with the separately stored CTG-only FRI one-variable Cox baseline. It writes comparison figures beneath the deep-learning result directory.


In [ ]:
from __future__ import annotations

import json
import re
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import auc, precision_recall_curve, roc_curve

# Change this one value to scale all plot text.
PLOT_TEXT_SCALE = 1.1
sns.set_theme(context="notebook", style="whitegrid", font_scale=PLOT_TEXT_SCALE)
# The packaged archive supplies the neural-model artifacts.
# Figures remain outside the archive so rerunning this notebook is read-only
# with respect to the packaged model files.
result_folder = "results_official_experiment"
REPO_ROOT = Path.cwd().resolve()
FINAL_SELECTION_DIR = REPO_ROOT / "final_model_selection"
FINAL_SELECTION_ARTIFACT_ROOT = FINAL_SELECTION_DIR / "artifacts"
if not FINAL_SELECTION_ARTIFACT_ROOT.is_dir():
    raise FileNotFoundError(f"Missing packaged model artifacts: {FINAL_SELECTION_ARTIFACT_ROOT}")
RESULT_DIR = FINAL_SELECTION_ARTIFACT_ROOT / result_folder
RESULT_DIRS = [RESULT_DIR]
PRETRAINED_CLASSIFIER_RESULT_DIR = FINAL_SELECTION_ARTIFACT_ROOT / "results_end_of_tracing_trained"
OUTPUT_DIR = REPO_ROOT / result_folder / "postprocessing_fri_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# This notebook is a baseline-only comparison. Set True to include the
# neural-model artifacts from RESULT_DIR in the same plots.
INCLUDE_NEURAL_RESULTS = True

# Packaged non-neural baseline artifacts are read from the portable archive.
BASELINE_ARTIFACTS = [
    {
        "directory": FINAL_SELECTION_ARTIFACT_ROOT / "results_ctg_baselines_1hz" / "ctg_only_fri_cox",
        "stem": "ctg_fri_cox",
        "family": "baseline",
        "run_id": "ctg_only_fri_cox",
        "model_type": "fri-cox",
        "imputation": "not_applicable",
    },
    {
        "directory": FINAL_SELECTION_ARTIFACT_ROOT / "results_ctg_baselines_1hz" / "xgboost_figo_ctg_cox",
        "stem": "xgboost_ctg",
        "family": "xgboost_baseline",
        "run_id": "xgboost_figo_ctg_cox",
        "model_type": "xgboost-figo-cox",
        "imputation": "not_applicable",
    },
]

# Change these as desired. The notebook retains all endpoint rows in
# tables.
SPLIT = "Test"
ROC_END_MINUTES = 0.0
# None includes the deep models and both non-neural baselines in one ROC panel.
ROC_FAMILY = None
FAMILY_LABEL = {
    "survival": "Threshold DeepHit",
    "marked": "Marked DeepHit",
    "marked_corn": "Marked DeepHit (CORN)",
    "competing": "Competing-risk DeepHit",
    "competing_threshold": "Binary competing-risk DeepHit",
    "classifier_inception_last_hour": "InceptionTime (last-hour classifier)",
    "baseline": "Rule-based",
    "xgboost_baseline": "XGBoost",
}
IMPUTATION_LABEL = {
    "ffill": "Forward fill",
    "not_applicable": "Not applicable",
}
MODEL_LABEL = {
    "inception": "InceptionTime",
    "inceptiontime-lstm": "InceptionTime-LSTM",
    "inception-lstm": "InceptionTime-LSTM (8h history)",
    "fri-cox": "CTG-only FRI (Cox)",
    "xgboost-figo-cox": "FIGO CTG features (XGBoost Cox)",
    "inception-classifier": "InceptionTime classifier",
}
COMPARE_BY = "model_type"
MODEL_TYPE_FILTER = None
IMPUTATION_FILTER = None
# Keep every report, table, legend, and ROC panel in this clinical
# comparison order.
REPORT_FAMILY_ORDER = ["baseline", "xgboost_baseline", "classifier_inception_last_hour", "competing", "marked"]
EXCLUDED_FAMILIES = {"competing_threshold"}
# One fixed palette for bar and ROC plots, keyed by model family.
FAMILY_LABEL_COLORS = dict(
    zip(
        [FAMILY_LABEL[family] for family in REPORT_FAMILY_ORDER],
        sns.color_palette("tab20", n_colors=len(REPORT_FAMILY_ORDER)),
    )
)
EXCLUDE_CORN_RESULTS = True
assert COMPARE_BY in {"configuration", "imputation", "model_type", None}
assert MODEL_TYPE_FILTER in {None, *MODEL_LABEL}
assert IMPUTATION_FILTER in {None, *IMPUTATION_LABEL}
assert isinstance(EXCLUDE_CORN_RESULTS, bool)
assert set(REPORT_FAMILY_ORDER).issubset(FAMILY_LABEL)
assert EXCLUDED_FAMILIES.issubset(FAMILY_LABEL)
assert ROC_FAMILY is None or ROC_FAMILY in FAMILY_LABEL
if COMPARE_BY == "configuration":
    assert MODEL_TYPE_FILTER is None and IMPUTATION_FILTER is None
    COMPARISON_COLUMN, COMPARISON_TITLE = (
        "configuration_label",
        "Model / input configuration",
    )
elif COMPARE_BY == "imputation":
    assert MODEL_TYPE_FILTER is not None and IMPUTATION_FILTER is None
    COMPARISON_COLUMN, COMPARISON_TITLE = (
        "imputation_label",
        "Imputation method",
    )
elif COMPARE_BY == "model_type":
    assert MODEL_TYPE_FILTER is None
    COMPARISON_COLUMN, COMPARISON_TITLE = (
        "model_label",
        "Model type (pooled across input methods)",
    )
else:
    assert MODEL_TYPE_FILTER is not None and IMPUTATION_FILTER is not None
    COMPARISON_COLUMN, COMPARISON_TITLE = None, "Fixed configuration"
POOL_INPUT_METHODS = COMPARE_BY == "model_type" and IMPUTATION_FILTER is None
assert RESULT_DIR.exists(), f"Missing results directory: {RESULT_DIR}"
print(f"Reading deep-learning results from {RESULT_DIR}")
print(
    "Baseline directories:",
    [str(item["directory"]) for item in BASELINE_ARTIFACTS],
)

In [ ]:
RUN_RE = re.compile(
    r"^(competing_threshold|survival|marked_corn|"
    r"marked|competing)_deephit_(.+)$"
)
MODEL_CLASS_TYPE = {
    "Inception_classifier": "inception",
    "InceptionTime_LSTM_classifier": "inceptiontime-lstm",
    "Inception_LSTM_deephit": "inception-lstm",
}
MODEL_RUN_RE = re.compile(
    r"(?:^|_)(inceptiontime-lstm|inception-lstm-deephit|"
    r"inception)(?:_|$)"
)
MODEL_RUN_TYPE = {
    "inceptiontime-lstm": "inceptiontime-lstm",
    "inception-lstm-deephit": "inception-lstm",
    "inception": "inception",
}


def artifact_stem(path: Path) -> str:
    for suffix in (
        "_history.csv",
        "_landmark_metrics.csv",
        "_landmark_predictions.csv",
    ):
        if path.name.endswith(suffix):
            return path.name.removesuffix(suffix)
    raise ValueError(f"Unrecognized run artifact: {path.name}")


def run_identity(path: Path) -> tuple[str, str, str]:
    stem = artifact_stem(path)
    match = RUN_RE.match(stem)
    if match is None:
        raise ValueError(f"Unrecognized run artifact: {path.name}")
    return match.group(1), match.group(2), stem


def manifest_metadata(result_dir: Path, stem: str) -> dict:
    path = result_dir / f"{stem}.json"
    if not path.is_file():
        return {}
    try:
        return json.loads(path.read_text())
    except (OSError, json.JSONDecodeError) as error:
        print(f"Ignoring unreadable manifest {path.name}: {error}")
        return {}


def fallback_metadata(run_id: str) -> tuple[str, str]:
    imputation_match = re.search(r"(?:^|_)impute-([^_]+)", run_id)
    model_match = MODEL_RUN_RE.search(run_id)
    if imputation_match is None or model_match is None:
        raise ValueError(
            f"Missing model or imputation metadata in run ID: {run_id}"
        )
    return MODEL_RUN_TYPE[model_match.group(1)], imputation_match.group(1)


def annotate(frame, *, family, run_id, model_type, imputation, source_file, source_path=None):
    frame = frame.copy()
    frame["family"] = family
    frame["family_label"] = FAMILY_LABEL.get(family, family)
    frame["family_order"] = (
        REPORT_FAMILY_ORDER.index(family)
        if family in REPORT_FAMILY_ORDER
        else len(REPORT_FAMILY_ORDER)
    )
    frame["imputation"] = imputation
    frame["imputation_label"] = IMPUTATION_LABEL.get(imputation, imputation)
    frame["model_type"] = model_type
    frame["model_label"] = MODEL_LABEL.get(model_type, model_type)
    frame["configuration_label"] = (
        frame["model_label"] + " — " + frame["imputation_label"]
    )
    frame["run_id"] = run_id
    frame["source_file"] = source_file
    frame["source_path"] = str(source_path) if source_path is not None else source_file
    return frame


def legend_title_left(leg):
    c = leg.get_children()[0]
    title = c.get_children()[0]
    hpack = c.get_children()[1]
    hpack.align = "center"
    c._children = [hpack]
    hpack._children = [title] + hpack.get_children()


def load_artifacts(suffix: str) -> pd.DataFrame:
    frames = []
    neural_paths = (
        sorted(
            path for result_dir in RESULT_DIRS
            for path in result_dir.glob(f"*{suffix}")
        )
        if INCLUDE_NEURAL_RESULTS else []
    )
    for path in neural_paths:
        try:
            source_result_dir = path.parent
            family, run_id, stem = run_identity(path)
            if family in EXCLUDED_FAMILIES or (
                EXCLUDE_CORN_RESULTS and family == "marked_corn"
            ):
                continue
            frame = pd.read_csv(path)
            metadata = manifest_metadata(source_result_dir, stem)
            model_class = metadata.get("model", {}).get("model_class")
            imputation = metadata.get("imputation", {}).get("method")
            if model_class in MODEL_CLASS_TYPE and imputation:
                model_type = MODEL_CLASS_TYPE[model_class]
            else:
                model_type, imputation = fallback_metadata(run_id)
            frames.append(
                annotate(
                    frame,
                    family=family,
                    run_id=run_id,
                    model_type=model_type,
                    imputation=imputation,
                    source_file=path.name,
                    source_path=path,
                )
            )
        except (ValueError, pd.errors.EmptyDataError) as error:
            print(f"Skipping {path.name}: {error}")
    if suffix != "_history.csv" and PRETRAINED_CLASSIFIER_RESULT_DIR.is_dir():
        classifier_path = PRETRAINED_CLASSIFIER_RESULT_DIR / f"Inception_classifier_last_hour{suffix}"
        if classifier_path.is_file():
            try:
                classifier_frame = pd.read_csv(classifier_path)
                frames.append(
                    annotate(
                        classifier_frame,
                        family="classifier_inception_last_hour",
                        run_id="pretrained_inception_last_hour",
                        model_type="inception-classifier",
                        imputation="ffill",
                        source_file=str(classifier_path.relative_to(REPO_ROOT)),
                        source_path=classifier_path,
                    )
                )
            except pd.errors.EmptyDataError:
                print(f"Skipping empty classifier artifact: {classifier_path}")

    for baseline in BASELINE_ARTIFACTS:
        if suffix == "_history.csv":
            continue
        path = baseline["directory"] / f"{baseline['stem']}{suffix}"
        if not path.is_file():
            print(f"Baseline artifact not found: {path}")
            continue
        try:
            frame = pd.read_csv(path)
            frames.append(
                annotate(
                    frame,
                    family=baseline["family"],
                    run_id=baseline["run_id"],
                    model_type=baseline["model_type"],
                    imputation=baseline["imputation"],
                    source_file=str(path.relative_to(REPO_ROOT)),
                )
            )
        except pd.errors.EmptyDataError:
            print(f"Skipping empty baseline artifact: {path}")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def select_comparison_subset(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame
    if MODEL_TYPE_FILTER is not None:
        frame = frame.loc[frame["model_type"].eq(MODEL_TYPE_FILTER)].copy()
    if IMPUTATION_FILTER is not None:
        frame = frame.loc[frame["imputation"].eq(IMPUTATION_FILTER)].copy()
    frame["comparison_value"] = (
        frame[COMPARISON_COLUMN]
        if COMPARISON_COLUMN is not None
        else "Selected configuration"
    )
    # Use concise family names in legends and ROC row labels. The separate
    # comparison_value column remains available in tables for provenance.
    frame["comparison_label"] = frame["family_label"]
    return frame.sort_values(
        ["family_order", "comparison_label", "source_file"], kind="stable"
    ).reset_index(drop=True)


histories = select_comparison_subset(load_artifacts("_history.csv"))
metrics = select_comparison_subset(load_artifacts("_landmark_metrics.csv"))
predictions = select_comparison_subset(
    load_artifacts("_landmark_predictions.csv")
)
if not histories.empty:
    histories["epoch"] = histories.groupby("source_file").cumcount() + 1

history_file_count = (
    histories.source_file.nunique() if not histories.empty else 0
)
metric_file_count = metrics.source_file.nunique() if not metrics.empty else 0
prediction_file_count = (
    predictions.source_file.nunique() if not predictions.empty else 0
)
print(f"History files: {history_file_count}")
print(f"Completed metric files: {metric_file_count}")
print(f"Completed prediction files: {prediction_file_count}")

In [ ]:
histories.columns

In [ ]:
# Training and validation loss curves: faint individual runs plus family mean.

if histories.empty:
    print("No history files found yet.")
else:
    available_losses = [
        column for column in ["train_loss", "val_loss"] if column in histories
    ]
    fig, axes = plt.subplots(
        1,
        len(available_losses),
        figsize=(7 * len(available_losses), 5),
        sharex=True,
    )
    axes = np.atleast_1d(axes)
    for axis, loss_column in zip(axes, available_losses):
        sns.lineplot(
            data=histories,
            x="epoch",
            y=loss_column,
            hue="comparison_label",
            units="run_id",
            estimator=None,
            alpha=0.18,
            legend=False,
            ax=axis,
        )
        sns.lineplot(
            data=histories,
            x="epoch",
            y=loss_column,
            hue="comparison_label",
            estimator="mean",
            errorbar=("ci", 95),
            linewidth=2.5,
            ax=axis,
            legend=False,
        )
        axis.set_title(loss_column.replace("_", " ").title())
        axis.set_ylabel("Loss")
    fig.suptitle(
        "Training curves by model family, type, and imputation", y=1.03
    )
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "loss_curves_by_family.pdf", dpi=220, bbox_inches="tight"
    )
    plt.show()

In [ ]:
# Metric tables retain individual-run CIs and aggregate completed runs
# by mean and standard deviation.
METRIC_LABEL = {
    "antolini_cindex": "Cause-specific Antolini concordance index",
    "dynamic_auc": "Dynamic AUROC",
    "auroc_acidemic_delivery_by_horizon": (
        "AUROC: acidemic delivery by horizon"
    ),
    "auroc_acidemic_delivery_given_delivery": (
        "AUROC: acidemia among deliveries in horizon"
    ),
    "auroc_end_landmark": "AUROC before tracing end",
}


def nice_value(s):
    return (
        f"{s['value_mean']:.2f} ("
        f"{s['mean_run_ci_lower']:.2f} - "
        f"{s['mean_run_ci_upper']:.2f})"
    )


def threshold_value(label: str) -> float:
    match = re.search(r"([0-9]+(?:\.[0-9]+)?)$", str(label))
    return float(match.group(1)) if match else np.nan


if metrics.empty:
    print("No completed landmark metric reports found yet.")
    metric_summary = pd.DataFrame()
else:
    metric_rows = metrics.loc[metrics["target"].ne("delivery")].copy()
    metric_rows["threshold"] = metric_rows["target"].map(threshold_value)
    metric_rows["metric_label"] = (
        metric_rows["metric"].map(METRIC_LABEL).fillna(metric_rows["metric"])
    )
    # A bootstrap resample can be estimable even if its full-cohort point
    # estimate is not. Summarize CIs only for finite point estimates.
    metric_rows["ci_lower_if_estimable"] = metric_rows["ci_lower"].where(
        metric_rows["value"].notna()
    )
    metric_rows["ci_upper_if_estimable"] = metric_rows["ci_upper"].where(
        metric_rows["value"].notna()
    )
    # Horizon is part of a metric's identity: e.g., the 1-, 4-, and 8-hour
    # AUCs must never be averaged together.
    group_columns = [
        "split",
        "family_label",
        "model_type",
        "model_label",
        "imputation",
        "imputation_label",
        "comparison_value",
        "comparison_label",
        "target",
        "threshold",
        "metric",
        "metric_label",
        "landmark_seconds_since_start",
        "prediction_horizon_seconds",
        "minutes_before_delivery",
    ]
    if POOL_INPUT_METHODS:
        group_columns = [
            column
            for column in group_columns
            if column not in {"imputation", "imputation_label"}
        ]
    metric_summary = metric_rows.groupby(
        group_columns, dropna=False, as_index=False
    ).agg(
        value_mean=("value", "mean"),
        value_sd=("value", "std"),
        completed_runs=("run_id", "nunique"),
        estimable_runs=("value", "count"),
        mean_run_ci_lower=("ci_lower_if_estimable", "mean"),
        mean_run_ci_upper=("ci_upper_if_estimable", "mean"),
        total_n=("n", "sum"),
        metric_error=(
            "error",
            lambda values: "; ".join(sorted(set(values.dropna().astype(str)))),
        ),
    )
    family_label_order = {
        FAMILY_LABEL[family]: index
        for index, family in enumerate(REPORT_FAMILY_ORDER)
    }
    metric_summary["family_order"] = (
        metric_summary["family_label"]
        .map(family_label_order)
        .fillna(len(REPORT_FAMILY_ORDER))
    )
    metric_summary = metric_summary.sort_values(
        ["metric", "threshold", "family_order", "comparison_value"],
        kind="stable",
    )

    # AUROC case/control counts are partly stored in the metric CSVs. Derive a
    # consistent count pair for every displayed metric from per-PID
    # predictions. Antolini uses acidemic events versus competing
    # non-acidemic deliveries; dynamic AUC uses events by horizon versus
    # patients still event-free at that horizon.
    def count_records(data, metric_name, cases, controls):
        records = data.copy()
        records["metric"] = metric_name
        records["metric_label"] = METRIC_LABEL[metric_name]
        records["threshold"] = records["target"].map(threshold_value)
        records["case_count"] = np.asarray(cases, dtype=int)
        records["control_count"] = np.asarray(controls, dtype=int)
        return records

    count_frames = []
    if not predictions.empty:
        prediction_rows = predictions.loc[
            predictions["target"].ne("delivery")
        ].copy()
        start_rows = prediction_rows.loc[
            prediction_rows["landmark_seconds_since_start"].notna()
        ].copy()
        if not start_rows.empty:
            harrell_rows = start_rows.drop_duplicates(
                [
                    "run_id",
                    "split",
                    "target",
                    "PID",
                    "landmark_seconds_since_start",
                ]
            )
            harrell_horizons = metric_rows.loc[
                metric_rows["metric"].eq("antolini_cindex"),
                [
                    "run_id",
                    "split",
                    "target",
                    "landmark_seconds_since_start",
                    "prediction_horizon_seconds",
                ],
            ].drop_duplicates()
            harrell_rows = harrell_rows.drop(
                columns="prediction_horizon_seconds"
            ).merge(
                harrell_horizons,
                on=[
                    "run_id",
                    "split",
                    "target",
                    "landmark_seconds_since_start",
                ],
                how="inner",
            )
            count_frames.append(
                count_records(
                    harrell_rows,
                    "antolini_cindex",
                    harrell_rows["event"].eq(1),
                    harrell_rows["event"].eq(0),
                )
            )
            count_frames.append(
                count_records(
                    start_rows,
                    "dynamic_auc",
                    start_rows["event_within_horizon"].eq(1),
                    start_rows["remaining_seconds"].gt(
                        start_rows["prediction_horizon_seconds"]
                    ),
                )
            )
            count_frames.append(
                count_records(
                    start_rows,
                    "auroc_acidemic_delivery_by_horizon",
                    start_rows["event_within_horizon"].eq(1),
                    start_rows["event_within_horizon"].eq(0),
                )
            )
            count_frames.append(
                count_records(
                    start_rows,
                    "auroc_acidemic_delivery_given_delivery",
                    start_rows["event_within_horizon"].eq(1),
                    start_rows["delivery_within_horizon"].eq(1)
                    & start_rows["event_within_horizon"].eq(0),
                )
            )
        end_rows = prediction_rows.loc[
            prediction_rows["minutes_before_delivery"].notna()
        ].copy()
        if not end_rows.empty:
            count_frames.append(
                count_records(
                    end_rows,
                    "auroc_end_landmark",
                    end_rows["event"].eq(1),
                    end_rows["event"].eq(0),
                )
            )
    # if count_frames:
    #     count_rows = pd.concat(count_frames, ignore_index=True)
    #     per_run_counts = count_rows.groupby(
    #         group_columns + ["run_id"], dropna=False, as_index=False
    #     ).agg(
    #         case_count=("case_count", "sum"),
    #         control_count=("control_count", "sum"),
    #     )
    #     count_summary = per_run_counts.groupby(
    #         group_columns, dropna=False, as_index=False
    #     ).agg(
    #         cases_mean=("case_count", "mean"),
    #         controls_mean=("control_count", "mean"),
    #     )
    #     metric_summary = metric_summary.merge(
    #         count_summary, on=group_columns, how="left"
    #     )
    # else:
    #     metric_summary["cases_mean"] = np.nan
    #     metric_summary["controls_mean"] = np.nan
    if count_frames:
        count_rows = pd.concat(count_frames, ignore_index=True)
        per_run_counts = count_rows.groupby(
            group_columns + ["run_id"], dropna=False, as_index=False
        ).agg(
            case_count=("case_count", "sum"),
            control_count=("control_count", "sum"),
        )
        count_summary = per_run_counts.groupby(
            group_columns, dropna=False, as_index=False
        ).agg(
            cases_mean=("case_count", "mean"),
            controls_mean=("control_count", "mean"),
        )
        metric_summary = metric_summary.merge(
            count_summary, on=group_columns, how="left"
        )

        # Plot counts are unique cohort totals, not averages over repeated runs
        # or model families.
        cohort_count_columns = [
            "split",
            "target",
            "threshold",
            "metric",
            "metric_label",
            "landmark_seconds_since_start",
            "prediction_horizon_seconds",
            "minutes_before_delivery",
        ]
        cohort_totals = (
            count_rows.drop_duplicates(cohort_count_columns + ["PID"])
            .groupby(cohort_count_columns, dropna=False, as_index=False)
            .agg(
                cases_total=("case_count", "sum"),
                controls_total=("control_count", "sum"),
            )
        )
        metric_summary = metric_summary.merge(
            cohort_totals, on=cohort_count_columns, how="left"
        )
    else:
        metric_summary["cases_mean"] = np.nan
        metric_summary["controls_mean"] = np.nan
        metric_summary["cases_total"] = np.nan
        metric_summary["controls_total"] = np.nan
    metric_summary.to_csv(OUTPUT_DIR / "metric_summary.csv", index=False)

    for metric_name, table_name in [
        ("antolini_cindex", "antolini_concordance_table.csv"),
        ("dynamic_auc", "dynamic_auroc_table.csv"),
        (
            "auroc_acidemic_delivery_by_horizon",
            "acidemic_delivery_horizon_auroc_table.csv",
        ),
        (
            "auroc_acidemic_delivery_given_delivery",
            "horizon_delivery_auroc_table.csv",
        ),
        ("auroc_end_landmark", "end_auroc_table.csv"),
    ]:
        table = metric_summary.query(
            "split == @SPLIT and metric == @metric_name"
        ).copy()
        table.to_csv(OUTPUT_DIR / table_name, index=False)
        print(f"\n{METRIC_LABEL[metric_name]} ({SPLIT})")
        if table.empty:
            print("  No completed rows yet.")
        else:
            if metric_name == "auroc_end_landmark":
                time_cols = ["minutes_before_delivery"]
            else:
                table["minutes_since_start"] = (
                    table["landmark_seconds_since_start"] / 60
                ).astype("int")
                table["Time Since Start (H)"] = (
                    table["landmark_seconds_since_start"] / 3600
                ).astype("int")
                table["Prediction Horizon (H)"] = (
                    table["prediction_horizon_seconds"] / 3600
                )
                time_cols = ["Time Since Start (H)", "Prediction Horizon (H)"]
            display(table.columns)
            table["Mean Value (95% CI)"] = table[
                ["value_mean", "mean_run_ci_lower", "mean_run_ci_upper"]
            ].apply(nice_value, axis=1)
            display(
                (
                    table
                    # .set_index(['target',time_col,'family_label'])
                    .set_index(
                        time_cols
                        + ["family_label", "comparison_value", "target"]
                    )[
                        # [['total_n', 'Mean Value (95% CI)']]
                        "Mean Value (95% CI)"
                    ].unstack()
                )
            )

In [ ]:
# Landmark figures use stored run-level CI bounds, not a second seaborn
# bootstrap.
def plot_landmark_summary(
    data,
    *,
    x_column,
    row_column,
    title,
    x_label,
    y_label,
    filename_prefix,
    column_title=None,
):
    """Write faceted bar and line summaries with asymmetric 95% CI bars."""
    row_values = list(data[row_column].dropna().unique())
    col_values = sorted(data["threshold"].dropna().unique())
    # Allocate space only to observed family/model combinations. A Cartesian
    # family x model grid creates mostly empty slots and visually thin bars
    # when a single FRI baseline is added to several deep-learning families.
    series = (
        data[["family_label", "comparison_value", "family_order"]]
        .drop_duplicates()
        .sort_values(["family_order", "comparison_value"], kind="stable")
        .copy()
    )
    series["label"] = (
        series["family_label"] + " — " + series["comparison_value"]
    )
    series_keys = list(
        series[["family_label", "comparison_value"]].itertuples(
            index=False, name=None
        )
    )
    palette = {
        key: FAMILY_LABEL_COLORS.get(key[0], "0.5") for key in series_keys
    }

    def y_axis_label(row_label):
        lines = [textwrap.fill(str(row_label), width=26)]
        if y_label:
            lines.append(textwrap.fill(y_label, width=26))
        return "\n\n".join(lines)

    def facet_values(subset, family, comparison, x_values):
        values = (
            subset.loc[
                subset["family_label"].eq(family)
                & subset["comparison_value"].eq(comparison)
            ]
            .set_index(x_column)
            .reindex(x_values)
        )
        mean = values["value_mean"].to_numpy(float)
        lower = values["mean_run_ci_lower"].to_numpy(float)
        upper = values["mean_run_ci_upper"].to_numpy(float)
        return mean, lower, upper

    # Write one compact figure per metric, with case/control counts below.
    # A symlog scale keeps rare acidemic cases visible beside controls.
    for row_index, row_value in enumerate(row_values):
        fig, axes = plt.subplots(
            2,
            len(col_values),
            figsize=(5 * len(col_values), 5.2),
            squeeze=False,
            sharex="col",
            gridspec_kw={"height_ratios": [3, 1]},
        )
        row_x_values = sorted(
            data.loc[data[row_column].eq(row_value), x_column]
            .dropna()
            .unique(),
            reverse=x_column == "minutes_before_delivery",
        )
        x_positions = np.arange(len(row_x_values))
        for col_index, threshold in enumerate(col_values):
            axis = axes[0, col_index]
            count_axis = axes[1, col_index]
            subset = data.loc[
                data[row_column].eq(row_value)
                & data["threshold"].eq(threshold)
            ]
            active_series_keys = [
                key
                for key in series_keys
                if subset.loc[
                    subset["family_label"].eq(key[0])
                    & subset["comparison_value"].eq(key[1]),
                    "value_mean",
                ]
                .notna()
                .any()
            ]
            width = 0.8 / max(len(active_series_keys), 1)
            for series_index, (family, comparison) in enumerate(
                active_series_keys
            ):
                mean, lower, upper = facet_values(
                    subset, family, comparison, row_x_values
                )
                valid = np.isfinite(mean)
                ci_valid = valid & np.isfinite(lower) & np.isfinite(upper)
                label = family
                positions = x_positions - 0.4 + (series_index + 0.5) * width
                if valid.any():
                    axis.bar(
                        positions[valid],
                        mean[valid],
                        width=width,
                        color=palette[(family, comparison)],
                        label=label,
                    )
                axis.errorbar(
                    positions[ci_valid],
                    mean[ci_valid],
                    yerr=[
                        mean[ci_valid] - lower[ci_valid],
                        upper[ci_valid] - mean[ci_valid],
                    ],
                    fmt="none",
                    ecolor="black",
                    capsize=1,
                    linewidth=1,
                )
            axis.set_xticks(x_positions)
            axis.tick_params(axis="x", labelbottom=False)
            axis.set_ylim(0, 1)
            axis.set_title(column_title or f"pH < {threshold:g}", fontsize=10 * PLOT_TEXT_SCALE)
            if col_index == 0:
                axis.set_ylabel(y_axis_label(row_value), labelpad=10)

            # counts = (
            #     subset.groupby(x_column, as_index=False)
            #     .agg(
            #         cases_mean=("cases_mean", "median"),
            #         controls_mean=("controls_mean", "median"),
            #     )
            #     .set_index(x_column)
            #     .reindex(row_x_values)
            # )
            # cases = counts["cases_mean"].to_numpy(float)
            # controls = counts["controls_mean"].to_numpy(float)
            counts = (
                subset.groupby(x_column, as_index=False)
                .agg(
                    cases_total=("cases_total", "max"),
                    controls_total=("controls_total", "max"),
                )
                .set_index(x_column)
                .reindex(row_x_values)
            )
            cases = counts["cases_total"].to_numpy(float)
            controls = counts["controls_total"].to_numpy(float)
            count_width = 0.36
            control_bars = count_axis.bar(
                x_positions - count_width / 2,
                controls,
                width=count_width,
                color="0.30",
                label="Controls",
            )
            case_bars = count_axis.bar(
                x_positions + count_width / 2,
                cases,
                width=count_width,
                color="#c44e52",
                label="Acidemic cases",
            )
            for bars, values in [(control_bars, controls), (case_bars, cases)]:
                labels = [
                    "" if not np.isfinite(value) else f"{value:.0f}"
                    for value in values
                ]
                count_axis.bar_label(
                    bars,
                    labels=labels,
                    label_type="edge",
                    padding=1,
                    fontsize=9 * PLOT_TEXT_SCALE,
                )
            count_axis.set_yscale("symlog", linthresh=1)
            count_axis.set_xticks(
                x_positions, [f"{value:g}" for value in row_x_values]
            )
            count_axis.grid(axis="y", which="both", alpha=0.35)
            count_axis.set_xlabel(x_label)
            if col_index == 0:
                count_axis.set_ylabel(
                    "Cases / controls\n(symlog)", labelpad=10
                )
            if col_index == 3:
                count_axis.legend(loc="best", ncols=2, fontsize="small")

        handles, labels = [], []
        for axis in axes[0].flat:
            for handle, label in zip(*axis.get_legend_handles_labels()):
                if label not in labels:
                    handles.append(handle)
                    labels.append(label)
        leg = fig.legend(
            handles,
            labels,
            title="Model",
            loc="lower center",
            ncol=max(len(handles), 1),
            bbox_to_anchor=(0.5, -0.02),
        )
        legend_title_left(leg)
        fig.suptitle(f"{title}: {row_value} ({SPLIT})", y=1.01)
        fig.tight_layout(rect=(0, 0.05, 1.0, 1.0))
        metric_slug = re.sub(r"[^a-z0-9]+", "_", str(row_value).lower()).strip(
            "_"
        )
        sns.despine()
        fig.savefig(
            OUTPUT_DIR / f"{filename_prefix}_{metric_slug}_bars.pdf",
            dpi=300,
            bbox_inches="tight",
        )
        plt.show()


if metric_summary.empty:
    print("No metric summary available.")
else:
    start_metrics = metric_summary.loc[
        metric_summary["metric"].isin(
            [
                "antolini_cindex",
                "dynamic_auc",
                "auroc_acidemic_delivery_by_horizon",
                "auroc_acidemic_delivery_given_delivery",
            ]
        )
        & metric_summary["landmark_seconds_since_start"].notna()
        & (metric_summary["split"] == SPLIT)
    ].copy()
    start_metrics["hours_since_start"] = (
        start_metrics["landmark_seconds_since_start"] / 3600
    )
    start_metrics["prediction_horizon_hours"] = (
        start_metrics["prediction_horizon_seconds"] / 3600
    )
    start_metrics["metric_horizon_label"] = np.where(
        start_metrics["prediction_horizon_hours"].notna(),
        start_metrics["metric_label"]
        + " (h="
        + start_metrics["prediction_horizon_hours"].map("{:g}".format)
        + "h)",
        start_metrics["metric_label"],
    )
    if not start_metrics.empty:
        plot_landmark_summary(
            start_metrics,
            x_column="hours_since_start",
            row_column="metric_horizon_label",
            title="Start-landmark metrics",
            x_label="Hours since tracing start",
            y_label="",
            filename_prefix="start_landmark_metric",
        )

    end_metrics = metric_summary.loc[
        (metric_summary["metric"] == "auroc_end_landmark")
        & metric_summary["minutes_before_delivery"].notna()
        & (metric_summary["split"] == SPLIT)
    ].copy()
    if not end_metrics.empty:
        plot_landmark_summary(
            end_metrics,
            x_column="minutes_before_delivery",
            row_column="metric_label",
            title="End-landmark AUROC",
            x_label="Minutes before tracing end",
            y_label="",
            filename_prefix="end_landmark_auroc",
        )

In [ ]:
# Macro-pH figures use the macro_pH rows reported by the evaluators.
# These are unweighted macro scores across the available pH thresholds.
#
# Macro figure layout is deliberately independent of plot_landmark_summary.
# Adjust these values without changing the threshold-specific figures above.
# In MACRO_AXES_ADJUST, 'left' and 'right' control the axes width.
MACRO_PANEL_WIDTH = 4.6
MACRO_FIG_HEIGHT = 5.2
# Legends on side
MACRO_AXES_ADJUST = {
    "left": 0.10,
    "right": 0.78,
    "bottom": 0.12,
    "top": 0.84,
    "hspace": 0.18,
}
MACRO_LEGEND_KWARGS = {
    "loc": "center left",
    "bbox_to_anchor": (0.79, 0.70),
    "fontsize": "small",
    "title_fontsize": "small",
    "ncol": 1,
}

# Legends on bottom
MACRO_AXES_ADJUST = {
    "left": 0.10,
    "right": 0.97,
    "bottom": 0.16,
    "top": 0.81,
    "hspace": 0.18,
}
MACRO_LEGEND_KWARGS = {
    "loc": "lower center",
    "bbox_to_anchor": (0.5, -0.02),
    "fontsize": "small",
    "title_fontsize": "small",
    "ncol": 5,
}

def plot_macro_ph_summary(
    data,
    *,
    x_column,
    row_column,
    title,
    x_label,
    filename_prefix,
    column_column=None,
    column_label=None,
):
    """Write a macro-pH figure per metric with forecast-horizon columns."""
    row_values = list(data[row_column].dropna().unique())
    # Allocate space only to observed family/model combinations. A Cartesian
    # family x model grid creates mostly empty slots and visually thin bars
    # when a single FRI baseline is added to several deep-learning families.
    series = (
        data[["family_label", "comparison_value", "family_order"]]
        .drop_duplicates()
        .sort_values(["family_order", "comparison_value"], kind="stable")
        .copy()
    )
    series["label"] = (
        series["family_label"] + " — " + series["comparison_value"]
    )
    series_keys = list(
        series[["family_label", "comparison_value"]].itertuples(
            index=False, name=None
        )
    )
    palette = {
        key: FAMILY_LABEL_COLORS.get(key[0], "0.5") for key in series_keys
    }

    for row_value in row_values:
        row_subset = data.loc[data[row_column].eq(row_value)]
        column_values = (
            sorted(row_subset[column_column].dropna().unique())
            if column_column
            else [None]
        )
        show_horizon_columns = (
            column_column is not None and len(column_values) > 1
        )
        fig, axes = plt.subplots(
            2,
            len(column_values),
            figsize=(MACRO_PANEL_WIDTH * len(column_values), MACRO_FIG_HEIGHT),
            # figsize=(9, 5.4),
            squeeze=False,
            sharex="col",
            gridspec_kw={"height_ratios": [3, 1]},
        )
        for column_index, column_value in enumerate(column_values):
            axis, count_axis = axes[:, column_index]
            subset = row_subset.copy()
            if show_horizon_columns:
                subset = subset.loc[subset[column_column].eq(column_value)]
                axis.set_title(
                    f"{column_label}: {column_value:g} h", fontsize=10 * PLOT_TEXT_SCALE
                )
            x_values = sorted(
                subset[x_column].dropna().unique(),
                reverse=x_column == "minutes_before_delivery",
            )
            x_positions = np.arange(len(x_values))
            active_series_keys = [
                key
                for key in series_keys
                if subset.loc[
                    subset["family_label"].eq(key[0])
                    & subset["comparison_value"].eq(key[1]),
                    "value_mean",
                ]
                .notna()
                .any()
            ]
            width = 0.8 / max(len(active_series_keys), 1)
            for series_index, (family, comparison) in enumerate(
                active_series_keys
            ):
                values = (
                    subset.loc[
                        subset["family_label"].eq(family)
                        & subset["comparison_value"].eq(comparison)
                    ]
                    .set_index(x_column)
                    .reindex(x_values)
                )
                mean = values["value_mean"].to_numpy(float)
                lower = values["mean_run_ci_lower"].to_numpy(float)
                upper = values["mean_run_ci_upper"].to_numpy(float)
                valid = np.isfinite(mean)
                ci_valid = valid & np.isfinite(lower) & np.isfinite(upper)
                positions = x_positions - 0.4 + (series_index + 0.5) * width
                label = family
                if valid.any():
                    axis.bar(
                        positions[valid],
                        mean[valid],
                        width=width,
                        color=palette[(family, comparison)],
                        label=label,
                    )
                axis.errorbar(
                    positions[ci_valid],
                    mean[ci_valid],
                    yerr=[
                        mean[ci_valid] - lower[ci_valid],
                        upper[ci_valid] - mean[ci_valid],
                    ],
                    fmt="none",
                    ecolor="black",
                    capsize=1,
                    linewidth=1,
                )
            axis.set_ylim(0, 1)
            axis.grid(axis="y", alpha=0.35)
            if column_index == 0:
                axis.set_ylabel(
                    textwrap.fill(str(row_value), width=26), labelpad=10
                )

            # counts = (
            #     subset.groupby(x_column, as_index=False)
            #     .agg(
            #         cases_mean=("cases_mean", "median"),
            #         controls_mean=("controls_mean", "median"),
            #     )
            #     .set_index(x_column)
            #     .reindex(x_values)
            # )
            # cases = counts["cases_mean"].to_numpy(float)
            # controls = counts["controls_mean"].to_numpy(float)
            counts = (
                subset.groupby(x_column, as_index=False)
                .agg(
                    cases_total=("cases_total", "max"),
                    controls_total=("controls_total", "max"),
                )
                .set_index(x_column)
                .reindex(x_values)
            )

            cases = counts["cases_total"].to_numpy(float)
            controls = counts["controls_total"].to_numpy(float)
            count_width = 0.36
            control_bars = count_axis.bar(
                x_positions - count_width / 2,
                controls,
                width=count_width,
                color="0.30",
                label="Controls",
            )
            case_bars = count_axis.bar(
                x_positions + count_width / 2,
                cases,
                width=count_width,
                color="#c44e52",
                label="Acidemic cases",
            )
            for bars, bar_values in [
                (control_bars, controls),
                (case_bars, cases),
            ]:
                labels = [
                    "" if not np.isfinite(value) else f"{value:.0f}"
                    for value in bar_values
                ]
                count_axis.bar_label(
                    bars,
                    labels=labels,
                    label_type="edge",
                    padding=1,
                    fontsize=9 * PLOT_TEXT_SCALE,
                )
            count_axis.set_yscale("symlog", linthresh=1)
            count_axis.set_xticks(
                x_positions, [f"{value:g}" for value in x_values]
            )
            count_axis.set_xlabel(x_label)
            # count_axis.grid(False)
            count_axis.grid(axis="y", which="both", alpha=0.35)
            count_axis.spines["right"].set_visible(False)
            count_axis.spines["top"].set_visible(False)
            if column_index == 0:
                count_axis.set_ylabel(
                    "Cases / controls\n(symlog)", labelpad=10
                )

            if column_index == len(column_values) - 1:
                count_axis.legend(loc="best", fontsize="small", ncols=2)

        handles, labels = [], []
        # Legends on side
        # for axis in axes[0]:
        #     for handle, label in zip(*axis.get_legend_handles_labels()):
        #         if label not in labels:
        #             handles.append(handle)
        #             labels.append(label)
        # fig.legend(handles, labels, title="Model", **MACRO_LEGEND_KWARGS)
        # fig.suptitle(f"{title}: {row_value} ({SPLIT})", y=0.96)
        # fig.subplots_adjust(**MACRO_AXES_ADJUST)

        # Legends on top
        for axis in axes[0]:
            for handle, label in zip(*axis.get_legend_handles_labels()):
                if label not in labels:
                    handles.append(handle)
                    labels.append(label)
        leg = fig.legend(handles, labels, title="Model", **MACRO_LEGEND_KWARGS)
        legend_title_left(leg)
        fig.suptitle(f"{title}: {row_value} ({SPLIT})", y=1.03)
        fig.subplots_adjust(**MACRO_AXES_ADJUST)

        metric_slug = re.sub(r"[^a-z0-9]+", "_", str(row_value).lower()).strip(
            "_"
        )
        sns.despine()
        fig.savefig(
            OUTPUT_DIR / f"{filename_prefix}_{metric_slug}_bars.pdf",
            dpi=220,
            bbox_inches="tight",
        )
        plt.show()


if metric_summary.empty:
    print("No metric summary available.")
else:
    macro_metrics = metric_summary.loc[
        metric_summary["target"].eq("macro_pH")
        & metric_summary["metric"].isin(METRIC_LABEL)
        & metric_summary["split"].eq(SPLIT)
    ].copy()
    if macro_metrics.empty:
        print("No macro_pH metric rows available.")
    else:
        # macro_pH rows have no single case/control count. Report the median
        # count across threshold-specific rows contributing to each macro
        # score.
        count_keys = [
            "split",
            "family_label",
            "model_type",
            "model_label",
            "comparison_value",
            "comparison_label",
            "metric",
            "metric_label",
            "landmark_seconds_since_start",
            "prediction_horizon_seconds",
            "minutes_before_delivery",
        ]
        threshold_counts = metric_summary.loc[
            metric_summary["target"].ne("macro_pH")
            & metric_summary["threshold"].notna()
        ]

        # macro_counts = threshold_counts.groupby(
        #     count_keys, dropna=False, as_index=False
        # ).agg(
        #     cases_mean=("cases_mean", "median"),
        #     controls_mean=("controls_mean", "median"),
        # )
        # macro_metrics = macro_metrics.drop(
        #     columns=["cases_mean", "controls_mean"], errors="ignore"
        # ).merge(macro_counts, on=count_keys, how="left")
        
        macro_counts = threshold_counts.groupby(
            count_keys, dropna=False, as_index=False
        ).agg(
            cases_total=("cases_total", "max"),
            controls_total=("controls_total", "max"),
        )
        macro_metrics = macro_metrics.drop(
            columns=[
                "cases_mean",
                "controls_mean",
                "cases_total",
                "controls_total",
            ],
            errors="ignore",
        ).merge(
            macro_counts,
            on=count_keys,
            how="left",
        )

        macro_start_metrics = macro_metrics.loc[
            macro_metrics["landmark_seconds_since_start"].notna()
        ].copy()
        macro_start_metrics["hours_since_start"] = (
            macro_start_metrics["landmark_seconds_since_start"] / 3600
        )
        macro_start_metrics["prediction_horizon_hours"] = (
            macro_start_metrics["prediction_horizon_seconds"] / 3600
        )
        macro_start_metrics["metric_horizon_label"] = np.where(
            macro_start_metrics["prediction_horizon_hours"].notna(),
            macro_start_metrics["metric_label"]
            + " (h="
            + macro_start_metrics["prediction_horizon_hours"].map(
                "{:g}".format
            )
            + "h)",
            macro_start_metrics["metric_label"],
        )
        if not macro_start_metrics.empty:
            plot_macro_ph_summary(
                macro_start_metrics,
                x_column="hours_since_start",
                row_column="metric_label",
                title="Macro-pH start-landmark metrics",
                x_label="Hours since tracing start",
                filename_prefix="macro_ph_start_landmark_metric",
                column_column="prediction_horizon_hours",
                column_label="Forecast horizon",
            )

        macro_end_metrics = macro_metrics.loc[
            macro_metrics["metric"].eq("auroc_end_landmark")
            & macro_metrics["minutes_before_delivery"].notna()
        ].copy()
        if not macro_end_metrics.empty:
            plot_macro_ph_summary(
                macro_end_metrics,
                x_column="minutes_before_delivery",
                row_column="metric_label",
                title="Macro-pH end-landmark AUROC",
                x_label="Minutes before tracing end",
                filename_prefix="macro_ph_end_landmark_auroc",
            )

In [ ]:
from sklearn.metrics import roc_auc_score

# ROC panels use one end-relative horizon. Repeated runs are ensembled
# by mean risk per PID.
if predictions.empty:
    print("No completed landmark prediction reports found yet.")
else:
    roc_rows = predictions.loc[
        (predictions["split"] == SPLIT)
        & predictions["minutes_before_delivery"].eq(ROC_END_MINUTES)
        & predictions["end_landmark_risk"].notna()
        & predictions["target"].ne("delivery")
        & (
            predictions["family"].eq(ROC_FAMILY)
            if ROC_FAMILY is not None
            else True
        )
    ].copy()
    roc_rows["threshold"] = roc_rows["target"].map(threshold_value)
    thresholds = sorted(roc_rows["threshold"].dropna().unique())
    comparisons = list(
        roc_rows[["family_order", "comparison_label"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["family_order", "comparison_label"], kind="stable")[
            "comparison_label"
        ]
    )
    if not thresholds or not comparisons:
        print(
            f"No ROC rows at {ROC_END_MINUTES:g} minutes before tracing end."
        )
    else:
        # Thresholds occupy one row. Every panel overlays every method,
        # retaining the same color and marker for a method across thresholds.
        fig, axes = plt.subplots(
            1,
            len(thresholds),
            figsize=(4.5 * len(thresholds), 4.5),
            squeeze=False,
        )
        axes = axes[0]
        colors = {
            comparison: FAMILY_LABEL_COLORS.get(comparison, "0.5")
            for comparison in comparisons
        }
        marker_cycle = ["o", "s", "^", "D", "P", "X"]
        for col, threshold in enumerate(thresholds):
            axis = axes[col]
            auroc_handles = []
            auroc_labels = []
            for method_index, comparison in enumerate(comparisons):
                data = roc_rows.loc[
                    (roc_rows["threshold"] == threshold)
                    & (roc_rows["comparison_label"] == comparison)
                ]
                ensemble = data.groupby("PID", as_index=False).agg(
                    event=("event", "first"),
                    risk=("end_landmark_risk", "mean"),
                )
                if ensemble.empty or ensemble["event"].nunique() < 2:
                    continue
                fpr, tpr, _ = roc_curve(ensemble["event"], ensemble["risk"])
                auroc = roc_auc_score(ensemble["event"], ensemble["risk"])
                (line,) = axis.plot(
                    fpr,
                    tpr,
                    color=colors[comparison],
                    marker=marker_cycle[method_index % len(marker_cycle)],
                    markevery=max(1, len(fpr) // 8),
                    markersize=4,
                    linewidth=2,
                    label=comparison,
                )
                auroc_handles.append(line)
                auroc_labels.append(f"{auroc:.3f}")
            axis.plot([0, 1], [0, 1], "--", color="0.5", linewidth=1)
            axis.set(
                title=f"pH < {threshold:g}",
                xlabel="False positive rate",
                xlim=(0, 1),
                ylim=(0, 1),
            )
            if col == 0:
                axis.set_ylabel("True positive rate")
            axis.add_artist(
                axis.legend(
                    auroc_handles,
                    auroc_labels,
                    title="AUROC",
                    loc="lower right",
                    fontsize=7 * PLOT_TEXT_SCALE,
                    title_fontsize=7 * PLOT_TEXT_SCALE,
                    frameon=True,
                    framealpha=0.85,
                    handlelength=1.5,
                    borderpad=0.4,
                    labelspacing=0.3,
                )
            )
        handles, labels = axes[0].get_legend_handles_labels()
        leg = fig.legend(
            handles,
            labels,
            title="Model",
            loc="lower center",
            ncol=len(comparisons),
            bbox_to_anchor=(0.5, -0.02),
        )

        legend_title_left(leg)


        roc_family_label = (
            FAMILY_LABEL[ROC_FAMILY]
            if ROC_FAMILY is not None
            else "all model families"
        )
        fig.suptitle(
            f"ROC curves: {roc_family_label}, {SPLIT}, "
            f"{ROC_END_MINUTES:g} minutes before tracing end",
            y=1.01,
        )
        fig.tight_layout(rect=(0, 0.05, 1, 1.0))
        fig.savefig(
            OUTPUT_DIR
            / f"roc_{ROC_FAMILY or 'all'}_end_{ROC_END_MINUTES:g}_minutes.pdf",
            dpi=220,
            bbox_inches="tight",
        )
        plt.show()

In [ ]:
# Precision-recall panels mirror the ROC layout and styling.
if predictions.empty:
    print("No completed landmark prediction reports found yet.")
else:
    pr_rows = predictions.loc[
        (predictions["split"] == SPLIT)
        & predictions["minutes_before_delivery"].eq(ROC_END_MINUTES)
        & predictions["end_landmark_risk"].notna()
        & predictions["target"].ne("delivery")
        & (
            predictions["family"].eq(ROC_FAMILY)
            if ROC_FAMILY is not None
            else True
        )
    ].copy()
    pr_rows["threshold"] = pr_rows["target"].map(threshold_value)
    thresholds = sorted(pr_rows["threshold"].dropna().unique())
    comparisons = list(
        pr_rows[["family_order", "comparison_label"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["family_order", "comparison_label"], kind="stable")[
            "comparison_label"
        ]
    )
    if not thresholds or not comparisons:
        print(
            f"No precision-recall rows at {ROC_END_MINUTES:g} minutes "
            "before tracing end."
        )
    else:
        fig, axes = plt.subplots(
            1,
            len(thresholds),
            figsize=(4.5 * len(thresholds), 4.5),
            squeeze=False,
        )
        axes = axes[0]
        colors = {
            comparison: FAMILY_LABEL_COLORS.get(comparison, "0.5")
            for comparison in comparisons
        }
        marker_cycle = ["o", "s", "^", "D", "P", "X"]
        for col, threshold in enumerate(thresholds):
            axis = axes[col]
            threshold_rows = pr_rows.loc[pr_rows["threshold"].eq(threshold)]
            outcome = threshold_rows[["PID", "event"]].drop_duplicates("PID")
            prevalence = outcome["event"].mean()
            for method_index, comparison in enumerate(comparisons):
                data = threshold_rows.loc[
                    threshold_rows["comparison_label"].eq(comparison)
                ]
                ensemble = data.groupby("PID", as_index=False).agg(
                    event=("event", "first"),
                    risk=("end_landmark_risk", "mean"),
                )
                if ensemble.empty or ensemble["event"].nunique() < 2:
                    continue
                precision, recall, _ = precision_recall_curve(
                    ensemble["event"], ensemble["risk"]
                )
                axis.plot(
                    recall,
                    precision,
                    color=colors[comparison],
                    marker=marker_cycle[method_index % len(marker_cycle)],
                    markevery=max(1, len(recall) // 8),
                    markersize=4,
                    linewidth=2,
                    label=comparison,
                )
            if np.isfinite(prevalence):
                axis.axhline(
                    prevalence, color="0.35", linestyle=":", linewidth=1.5
                )
                axis.text(
                    0.98,
                    prevalence,
                    f"Prevalence = {prevalence:.3f}",
                    transform=axis.get_yaxis_transform(),
                    ha="right",
                    va="bottom",
                    color="0.25",
                    fontsize="small",
                )
            axis.set(
                title=f"pH < {threshold:g}",
                xlabel="Recall",
                xlim=(0, 1),
                ylim=(0, 1),
            )
            if col == 0:
                axis.set_ylabel("Precision")
        handles, labels = axes[0].get_legend_handles_labels()
        leg = fig.legend(
            handles,
            labels,
            title="Model",
            loc="lower center",
            ncol=len(comparisons),
            bbox_to_anchor=(0.5, -0.02),
        )
        legend_title_left(leg)
        pr_family_label = (
            FAMILY_LABEL[ROC_FAMILY]
            if ROC_FAMILY is not None
            else "all model families"
        )
        fig.suptitle(
            f"Precision-recall curves: {pr_family_label}, {SPLIT}, "
            f"{ROC_END_MINUTES:g} minutes before tracing end",
            y=1.01,
        )
        fig.tight_layout(rect=(0, 0.05, 1, 1))
        fig.savefig(
            OUTPUT_DIR
            / (
                f"precision_recall_{ROC_FAMILY or 'all'}_end_"
                f"{ROC_END_MINUTES:g}_minutes.pdf"
            ),
            dpi=220,
            bbox_inches="tight",
        )
        plt.show()

In [ ]:
# Delivery-timing comparisons adapted from the delivery-model notebook.
DELIVERY_METRIC_LABEL = {
    "antolini_cindex": "Delivery-time\nAntolini concordance index",
    "auroc_delivery_by_horizon": "Delivery-within-horizon\nAUROC",
    "auprc_delivery_by_horizon": "Delivery-within-horizon\nAUPRC",
}
delivery_group_columns = [
    "split",
    "family_label",
    "family_order",
    "model_type",
    "model_label",
    "imputation",
    "imputation_label",
    "comparison_value",
    "comparison_label",
    "metric",
    "metric_label",
    "landmark_seconds_since_start",
    "prediction_horizon_seconds",
    "outcome_followup_seconds",
]

if metrics.empty:
    delivery_summary = pd.DataFrame()
    print("No completed landmark metric reports found.")
else:
    delivery_rows = metrics.loc[
        metrics["target"].eq("delivery")
        & metrics["metric"].isin(DELIVERY_METRIC_LABEL)
    ].copy()
    delivery_rows["metric_label"] = delivery_rows["metric"].map(
        DELIVERY_METRIC_LABEL
    )
    delivery_rows["ci_lower_if_estimable"] = delivery_rows["ci_lower"].where(
        delivery_rows["value"].notna()
    )
    delivery_rows["ci_upper_if_estimable"] = delivery_rows["ci_upper"].where(
        delivery_rows["value"].notna()
    )
    delivery_summary = delivery_rows.groupby(
        delivery_group_columns, dropna=False, as_index=False
    ).agg(
        value_mean=("value", "mean"),
        value_sd=("value", "std"),
        completed_runs=("run_id", "nunique"),
        estimable_runs=("value", "count"),
        mean_run_ci_lower=("ci_lower_if_estimable", "mean"),
        mean_run_ci_upper=("ci_upper_if_estimable", "mean"),
        total_n=("n", "sum"),
        metric_error=(
            "error",
            lambda values: "; ".join(sorted(set(values.dropna().astype(str)))),
        ),
    )
    delivery_predictions = predictions.loc[
        predictions["target"].eq("delivery")
    ].copy()
    if not delivery_predictions.empty:
        delivery_counts = delivery_predictions.loc[
            delivery_predictions["landmark_seconds_since_start"].notna()
        ].copy()
        delivery_counts["metric"] = np.where(
            delivery_counts["prediction_horizon_seconds"].eq(172800),
            "antolini_cindex",
            "auroc_delivery_by_horizon",
        )
        # AUPRC and AUROC use the same delivery-within-horizon cohort.
        auprc_counts = delivery_counts.loc[
            delivery_counts["metric"].eq("auroc_delivery_by_horizon")
        ].copy()
        auprc_counts["metric"] = "auprc_delivery_by_horizon"
        delivery_counts = pd.concat(
            [delivery_counts, auprc_counts], ignore_index=True
        )
        delivery_counts["metric_label"] = delivery_counts["metric"].map(
            DELIVERY_METRIC_LABEL
        )
        delivery_counts["case_count"] = (
            delivery_counts["delivery_within_horizon"].eq(1).astype(int)
        )
        delivery_counts["control_count"] = (
            delivery_counts["delivery_within_horizon"].eq(0).astype(int)
        )
        per_run_counts = delivery_counts.groupby(
            delivery_group_columns + ["run_id"], dropna=False, as_index=False
        ).agg(
            cases_mean=("case_count", "sum"),
            controls_mean=("control_count", "sum"),
        )
        count_summary = per_run_counts.groupby(
            delivery_group_columns, dropna=False, as_index=False
        ).agg(
            cases_mean=("cases_mean", "mean"),
            controls_mean=("controls_mean", "mean"),
        )
        delivery_summary = delivery_summary.merge(
            count_summary, on=delivery_group_columns, how="left"
        )
    else:
        delivery_summary["cases_mean"] = np.nan
        delivery_summary["controls_mean"] = np.nan
    delivery_summary = delivery_summary.sort_values(
        [
            "metric",
            "prediction_horizon_seconds",
            "family_order",
            "comparison_value",
        ],
        kind="stable",
    )
    delivery_summary.to_csv(
        OUTPUT_DIR / "delivery_metric_summary.csv", index=False
    )


def plot_delivery_timing_summary(data):
    series = (
        data[["family_label", "comparison_value", "family_order"]]
        .drop_duplicates()
        .sort_values(["family_order", "comparison_value"], kind="stable")
    )
    series_keys = list(
        series[["family_label", "comparison_value"]].itertuples(
            index=False, name=None
        )
    )
    palette = {
        key: FAMILY_LABEL_COLORS.get(key[0], "0.5") for key in series_keys
    }
    for metric_label in data["metric_horizon_label"].dropna().unique():
        subset = data.loc[data["metric_horizon_label"].eq(metric_label)]
        x_values = sorted(subset["hours_since_start"].dropna().unique())
        x_positions = np.arange(len(x_values))
        fig, axes = plt.subplots(
            2,
            1,
            figsize=(9, 5.4),
            sharex=True,
            gridspec_kw={"height_ratios": [3, 1]},
        )
        axis, count_axis = axes
        active_series = [
            key
            for key in series_keys
            if subset.loc[
                subset["family_label"].eq(key[0])
                & subset["comparison_value"].eq(key[1]),
                "value_mean",
            ]
            .notna()
            .any()
        ]
        width = 0.8 / max(len(active_series), 1)
        for series_index, (family, comparison) in enumerate(active_series):
            values = (
                subset.loc[
                    subset["family_label"].eq(family)
                    & subset["comparison_value"].eq(comparison)
                ]
                .set_index("hours_since_start")
                .reindex(x_values)
            )
            mean = values["value_mean"].to_numpy(float)
            lower = values["mean_run_ci_lower"].to_numpy(float)
            upper = values["mean_run_ci_upper"].to_numpy(float)
            valid = np.isfinite(mean)
            ci_valid = valid & np.isfinite(lower) & np.isfinite(upper)
            positions = x_positions - 0.4 + (series_index + 0.5) * width
            if valid.any():
                axis.bar(
                    positions[valid],
                    mean[valid],
                    width=width,
                    color=palette[(family, comparison)],
                    label=family,
                )
            axis.errorbar(
                positions[ci_valid],
                mean[ci_valid],
                yerr=[
                    mean[ci_valid] - lower[ci_valid],
                    upper[ci_valid] - mean[ci_valid],
                ],
                fmt="none",
                ecolor="black",
                capsize=1,
                linewidth=1,
            )
        axis.set_ylim(0, 1)
        axis.set_ylabel(metric_label)
        axis.grid(axis="y", alpha=0.35)
        counts = (
            subset.groupby("hours_since_start", as_index=False)
            .agg(
                cases_mean=("cases_mean", "median"),
                controls_mean=("controls_mean", "median"),
            )
            .set_index("hours_since_start")
            .reindex(x_values)
        )
        count_width = 0.36
        controls = counts["controls_mean"].to_numpy(float)
        cases = counts["cases_mean"].to_numpy(float)
        control_bars = count_axis.bar(
            x_positions - count_width / 2,
            controls,
            width=count_width,
            color="0.30",
            label="Not delivered",
        )
        case_bars = count_axis.bar(
            x_positions + count_width / 2,
            cases,
            width=count_width,
            color="#4c72b0",
            label="Delivered",
        )
        for bars, values in [(control_bars, controls), (case_bars, cases)]:
            count_axis.bar_label(
                bars,
                labels=[
                    "" if not np.isfinite(value) else f"{value:.0f}"
                    for value in values
                ],
                label_type="edge",
                padding=1,
                fontsize=9 * PLOT_TEXT_SCALE,
            )
        count_axis.set_yscale("symlog", linthresh=1)
        count_axis.set_xticks(
            x_positions, [f"{value:g}" for value in x_values]
        )
        count_axis.tick_params(axis='y', labelsize=8)
        count_axis.set_xlabel("Hours since tracing start")
        count_axis.set_ylabel("Patients\n(symlog)")
        count_axis.grid(axis="y", which="both", alpha=0.35)
        count_leg = count_axis.legend(
            title="Cohort",
            loc="best",
            fontsize="small",
            title_fontsize="small",
            ncols=2,
        )
        legend_title_left(count_leg)
        handles, labels = axis.get_legend_handles_labels()
        unique_handles, unique_labels = [], []
        for handle, label in zip(handles, labels):
            if label not in unique_labels:
                unique_handles.append(handle)
                unique_labels.append(label)
        leg = fig.legend(
            unique_handles,
            unique_labels,
            title="Model",
            loc="lower center",
            bbox_to_anchor=(0.5, 0.80),
            ncol=3,
        )
        legend_title_left(leg)
        
        fig.suptitle(
            f"Delivery-timing comparison: {metric_label} ({SPLIT})", y=1.05
        )
        fig.tight_layout(rect=(0, 0, 1, 0.95))
        slug = re.sub(r"[^a-z0-9]+", "_", metric_label.lower()).strip("_")
        sns.despine()
        fig.savefig(
            OUTPUT_DIR / f"delivery_timing_{slug}_bars.pdf",
            dpi=300,
            bbox_inches="tight",
        )
        plt.show()

if delivery_summary.empty:
    print("No delivery metrics are available.")
else:
    delivery_start_metrics = delivery_summary.loc[
        delivery_summary["split"].eq(SPLIT)
        & delivery_summary["landmark_seconds_since_start"].notna()
    ].copy()
    delivery_start_metrics["hours_since_start"] = (
        delivery_start_metrics["landmark_seconds_since_start"] / 3600
    )
    delivery_start_metrics["prediction_horizon_hours"] = (
        delivery_start_metrics["prediction_horizon_seconds"] / 3600
    )
    delivery_start_metrics["metric_horizon_label"] = np.where(
        delivery_start_metrics["metric"].eq("antolini_cindex"),
        delivery_start_metrics["metric_label"],
        delivery_start_metrics["metric_label"]
        + " (h="
        + delivery_start_metrics["prediction_horizon_hours"].map("{:g}".format)
        + "h)",
    )
    plot_delivery_timing_summary(delivery_start_metrics)
display(delivery_summary.loc[
    delivery_summary["metric"].eq("auroc_delivery_by_horizon")
    & delivery_summary["prediction_horizon_seconds"].eq(3600)
    & delivery_summary["outcome_followup_seconds"].eq(3600)
    & delivery_summary["family_label"].isin(["Marked DeepHit", "XGBoost"]),
    [
        "family_label",
        "landmark_seconds_since_start",
        "value_mean",
        "mean_run_ci_lower",
        "mean_run_ci_upper",
    ],
].sort_values(["family_label", "landmark_seconds_since_start"]))

In [ ]:
# One-hour delivery-risk diagnostics for all four comparison methods.
DELIVERY_RISK_BIN_EDGES_HOURS = [0, 1, 4, 8, 16, 24, 48, np.inf]
DELIVERY_MAX_STRIP_POINTS = 600
DELIVERY_RISK_HORIZON_SECONDS = 60 * 60
delivery_risk_rows = predictions.loc[
    predictions["target"].eq("delivery")
    & predictions["split"].eq(SPLIT)
    & predictions["landmark_seconds_since_start"].notna()
    & predictions["prediction_horizon_seconds"].eq(
        DELIVERY_RISK_HORIZON_SECONDS
    )
    & predictions["horizon_risk"].notna()
].copy()
if delivery_risk_rows.empty:
    print("No one-hour delivery horizon-risk rows are available.")
else:
    delivery_risk_rows = delivery_risk_rows.drop_duplicates(
        ["run_id", "split", "PID", "landmark_seconds_since_start"]
    ).copy()
    delivery_risk_rows["remaining_hours"] = (
        delivery_risk_rows["remaining_seconds"] / 3600
    )
    delivery_risk_rows["remaining_time_band"] = pd.cut(
        delivery_risk_rows["remaining_hours"],
        bins=DELIVERY_RISK_BIN_EDGES_HOURS,
        labels=["≤1h", "1–4h", "4–8h", "8–16h", "16–24h", "24–48h", "≥48h"],
        include_lowest=True,
    )
    delivery_risk_rows["landmark_hours"] = (
        delivery_risk_rows["landmark_seconds_since_start"] / 3600
    )
    tie_summary = delivery_risk_rows.groupby(
        ["comparison_label", "landmark_hours"], dropna=False, as_index=False
    ).agg(
        n_scores=("horizon_risk", "size"),
        unique_scores=("horizon_risk", "nunique"),
    )
    tie_summary["near_tied_scores"] = (
        tie_summary["n_scores"] - tie_summary["unique_scores"]
    )
    tie_summary["near_tie_fraction"] = (
        tie_summary["near_tied_scores"] / tie_summary["n_scores"]
    )
    tie_summary.to_csv(
        OUTPUT_DIR / "delivery_1h_horizon_risk_ties.csv", index=False
    )
    DELIVERY_RISK_LANDMARK_HOURS = [1, 6, 12]
    landmark_mask = np.zeros(len(delivery_risk_rows), dtype=bool)
    for landmark_hours in DELIVERY_RISK_LANDMARK_HOURS:
        landmark_mask |= np.isclose(
            delivery_risk_rows["landmark_hours"], landmark_hours
        )
    plot_rows = delivery_risk_rows.loc[landmark_mask].copy()
    plot_rows["landmark_label"] = pd.Categorical(
        plot_rows["landmark_hours"].map(lambda value: f"{value:g}h"),
        categories=[f"{value:g}h" for value in DELIVERY_RISK_LANDMARK_HOURS],
        ordered=True,
    )
    if plot_rows.empty:
        print("No 1-, 6-, or 12-hour landmark rows are available.")
    else:
        landmark_palette = dict(
            zip(
                [f"{value:g}h" for value in DELIVERY_RISK_LANDMARK_HOURS],
                sns.color_palette("Set2", len(DELIVERY_RISK_LANDMARK_HOURS)),
            )
        )
        for comparison, subset in plot_rows.groupby(
            "comparison_label", dropna=False
        ):
            fig, axes = plt.subplots(
                1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [3, 2]}
            )
            left_plot_data = subset.loc[
                subset["remaining_time_band"].ne("≥48h")
            ].copy()
            strip_data = left_plot_data.sample(
                n=min(len(left_plot_data), DELIVERY_MAX_STRIP_POINTS),
                random_state=42,
            )
            sns.boxplot(
                data=left_plot_data,
                x="remaining_time_band",
                y="horizon_risk",
                hue="landmark_label",
                palette=landmark_palette,
                showfliers=False,
                ax=axes[0],
            )
            sns.stripplot(
                data=strip_data,
                x="remaining_time_band",
                y="horizon_risk",
                hue="landmark_label",
                palette=landmark_palette,
                dodge=True,
                alpha=0.22,
                size=1.5,
                legend=False,
                ax=axes[0],
            )
            axes[0].set(
                xlabel="Observed remaining time to delivery",
                ylabel="Predicted delivery risk within 1 hour",
                ylim=(0, 1),
            )
            axes[0].tick_params(axis="x") #, rotation=35)
            axes[0].grid(axis="y", alpha=0.35)
            axes[0].legend(title="Elapsed-time landmark")
            sns.kdeplot(
                data=subset,
                x="horizon_risk",
                hue="remaining_time_band",
                fill=True,
                common_norm=False,
                clip=(0, 1),
                ax=axes[1],
            )
            axes[1].set(
                xlabel="Predicted delivery risk within 1 hour",
                ylabel="Density",
                xlim=(0, 1),
            )
            axes[1].grid(axis="y", alpha=0.35)
            fig.suptitle(
                f"Delivery risk within 1 hour: {comparison}, "
                f"1-, 6-, and 12-hour landmarks ({SPLIT})",
                y=1.02,
            )
            fig.tight_layout()
            comparison_slug = re.sub(
                r"[^a-z0-9]+", "_", str(comparison).lower()
            ).strip("_")
            fig.savefig(
                OUTPUT_DIR
                / (
                    f"delivery_1h_horizon_risk_{comparison_slug}_"
                    "landmarks_1h_6h_12h.pdf"
                ),
                dpi=220,
                bbox_inches="tight",
            )
            plt.show()


In [ ]:
# Marked DeepHit calibration for one-hour delivery at each start landmark.
# Every patient has an observed delivery time, so the fixed one-hour outcome
# is fully observed in this retained-tracing cohort (no censoring correction).
CALIBRATION_HORIZON_SECONDS = 60 * 60
CALIBRATION_BINS = 10
marked_calibration_rows = predictions.loc[
    predictions["family"].eq("marked")
    & predictions["target"].eq("delivery")
    & predictions["split"].eq(SPLIT)
    & predictions["landmark_seconds_since_start"].notna()
    & predictions["prediction_horizon_seconds"].eq(CALIBRATION_HORIZON_SECONDS)
    & predictions["horizon_risk"].notna()
].copy()

if marked_calibration_rows.empty:
    print("No marked DeepHit one-hour delivery-risk rows are available.")
else:
    # Mean risk over completed repeated runs retains a single, PID-aligned
    # held-out prediction per landmark. Truth must agree across repetitions.
    marked_calibration_rows = marked_calibration_rows.groupby(
        ["PID", "landmark_seconds_since_start"], as_index=False
    ).agg(
        delivery_within_1h=("delivery_within_horizon", "first"),
        predicted_risk=("horizon_risk", "mean"),
        repeated_runs=("run_id", "nunique"),
    )
    calibration_summary_records = []
    calibration_bin_records = []
    landmarks = sorted(
        marked_calibration_rows["landmark_seconds_since_start"].unique()
    )
    fig, axes = plt.subplots(
        1, len(landmarks), figsize=(4.6 * len(landmarks), 4.5), squeeze=False
    )
    axes = axes[0]
    for axis, landmark_seconds in zip(axes, landmarks):
        data = marked_calibration_rows.loc[
            marked_calibration_rows["landmark_seconds_since_start"].eq(
                landmark_seconds
            )
        ].copy()
        truth = data["delivery_within_1h"].to_numpy(int)
        risk = data["predicted_risk"].to_numpy(float)
        clipped_risk = np.clip(risk, 1e-6, 1 - 1e-6)
        logit_risk = np.log(clipped_risk / (1 - clipped_risk))
        if np.unique(truth).size == 2:
            recalibration = LogisticRegression(
                C=1e6, solver="lbfgs", max_iter=1000
            ).fit(logit_risk.reshape(-1, 1), truth)
            calibration_intercept = float(recalibration.intercept_[0])
            calibration_slope = float(recalibration.coef_[0, 0])
        else:
            calibration_intercept = np.nan
            calibration_slope = np.nan
        prevalence = float(truth.mean())
        mean_risk = float(risk.mean())
        brier_score = float(np.mean((risk - truth) ** 2))
        eo_ratio = mean_risk / prevalence if prevalence > 0 else np.nan
        calibration_summary_records.append(
            {
                "split": SPLIT,
                "family": "marked",
                "landmark_seconds_since_start": landmark_seconds,
                "landmark_hours_since_start": landmark_seconds / 3600,
                "horizon_seconds": CALIBRATION_HORIZON_SECONDS,
                "n_patients": len(data),
                "events_within_1h": int(truth.sum()),
                "observed_prevalence": prevalence,
                "mean_predicted_risk": mean_risk,
                "expected_observed_ratio": eo_ratio,
                "brier_score": brier_score,
                "recalibration_intercept": calibration_intercept,
                "recalibration_slope": calibration_slope,
                "repeated_runs_ensembled": int(data["repeated_runs"].max()),
            }
        )
        bin_count = min(
            CALIBRATION_BINS, int(data["predicted_risk"].nunique())
        )
        if bin_count >= 2:
            data["risk_bin"] = pd.qcut(
                data["predicted_risk"], q=bin_count, duplicates="drop"
            )
            bins = data.groupby("risk_bin", observed=True, as_index=False).agg(
                mean_predicted_risk=("predicted_risk", "mean"),
                observed_frequency=("delivery_within_1h", "mean"),
                n=("delivery_within_1h", "size"),
            )
            bins["observed_se"] = np.sqrt(
                bins["observed_frequency"]
                * (1 - bins["observed_frequency"])
                / bins["n"]
            )
            bins["landmark_seconds_since_start"] = landmark_seconds
            bins["landmark_hours_since_start"] = landmark_seconds / 3600
            calibration_bin_records.extend(bins.to_dict("records"))
            axis.errorbar(
                bins["mean_predicted_risk"],
                bins["observed_frequency"],
                yerr=1.96 * bins["observed_se"],
                fmt="o-",
                color=FAMILY_LABEL_COLORS["Marked DeepHit"],
                capsize=2,
                label="Marked DeepHit",
            )
        axis.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            color="0.4",
            linewidth=1,
            label="Perfect calibration",
        )
        axis.set(
            title=f"{landmark_seconds / 3600:g}h landmark",
            xlabel="Mean predicted 1-hour delivery risk",
            xlim=(0, 1),
            ylim=(0, 1),
        )
        if axis is axes[0]:
            axis.set_ylabel("Observed 1-hour delivery frequency")
        axis.grid(alpha=0.35)
    handles, labels = axes[2].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        title="Model",
        loc="lower center",
        ncol=2,
        bbox_to_anchor=(0.5, -0.02),
    )
    fig.suptitle(
        f"Marked DeepHit: 1-hour delivery-risk calibration ({SPLIT})", y=1.02
    )
    fig.tight_layout(rect=(0, 0.10, 1, 1))
    fig.savefig(
        OUTPUT_DIR / "marked_deephit_delivery_1h_calibration.pdf",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()
    calibration_summary = pd.DataFrame(calibration_summary_records)
    calibration_bins = pd.DataFrame(calibration_bin_records)
    calibration_summary.to_csv(
        OUTPUT_DIR / "marked_deephit_delivery_1h_calibration_summary.csv",
        index=False,
    )
    calibration_bins.to_csv(
        OUTPUT_DIR / "marked_deephit_delivery_1h_calibration_bins.csv",
        index=False,
    )
    display(calibration_summary)

## Notes

- `value_mean` summarizes completed runs; it is not a pooled patient-level estimate. Individual-run bootstrap intervals remain in the source landmark metric files.
- ROC panels ensemble repeated completed runs by mean predicted risk for each PID.
- Standard threshold DeepHit runs contain one threshold each. Marked and competing-risk reports contribute all thresholds (and `macro_pH` rows, which are retained in `metric_summary.csv` but excluded from threshold-specific ROC panels).

## Paper-ready figures and primary AUROC table

## Paper-ready primary AUROC table

In [ ]:
# Paper-ready primary AUROC table.
# The existing notebook figure cells generate the paper figures.
TABLE_DIR = OUTPUT_DIR / 'tables'
TABLE_DIR.mkdir(parents=True, exist_ok=True)


def _latex_escape(value):
    return str(value).replace('\\', r'\textbackslash{}').replace('_', r'\_').replace('&', r'\&').replace('%', r'\%').replace('#', r'\#')


def write_primary_auroc_table():
    if metric_summary.empty:
        print('No metric summary is available for the LaTeX table.')
        return
    data = metric_summary.loc[
        metric_summary['split'].eq(SPLIT)
        & metric_summary['metric'].eq('auroc_acidemic_delivery_by_horizon')
        & ~np.isclose(metric_summary['prediction_horizon_seconds'], 12 * 3600)
        & ~np.isclose(metric_summary['prediction_horizon_seconds'], 48 * 3600)
    ].copy()
    if data.empty:
        print('No eligible primary AUROC rows are available for the LaTeX table.')
        return
    data['landmark_h'] = data['landmark_seconds_since_start'] / 3600
    data['horizon_h'] = data['prediction_horizon_seconds'] / 3600
    data = data.groupby(
        ['landmark_h', 'horizon_h', 'family_label', 'comparison_value', 'target'],
        as_index=False,
        dropna=False,
    ).agg(
        value_mean=('value_mean', 'mean'),
        ci_lower=('mean_run_ci_lower', 'mean'),
        ci_upper=('mean_run_ci_upper', 'mean'),
    )
    outcomes = sorted(
        data['target'].dropna().unique(),
        key=lambda target: (target != 'macro_pH', threshold_value(target)),
    )
    best_values = data.groupby(
        ['landmark_h', 'horizon_h', 'target'], dropna=False,
    )['value_mean'].max()

    # Build the canonical table with landmark/outcome rows and model
    # columns. This is the paper layout emitted below.
    table_family_order = [
        'competing',
        'classifier_inception_last_hour',
        'marked',
        'baseline',
        'xgboost_baseline',
    ]
    model_keys = (
        data[['family_label', 'comparison_value']]
        .drop_duplicates()
        .assign(
            family_order=lambda frame: frame['family_label'].map(
                {
                    FAMILY_LABEL[family]: index
                    for index, family in enumerate(table_family_order)
                }
            ).fillna(len(table_family_order))
        )
        .sort_values(['family_order', 'family_label', 'comparison_value'], kind='stable')
        .reset_index(drop=True)
        .rename_axis('model_id')
        .reset_index()
    )
    data = data.merge(
        model_keys[['model_id', 'family_label', 'comparison_value']],
        on=['family_label', 'comparison_value'],
        how='left',
        validate='many_to_one',
    )
    data['cell'] = data.apply(
        lambda record: (
            f"{record['value_mean']:.3f} ({record['ci_lower']:.3f}--{record['ci_upper']:.3f})"
            if np.isfinite(record['ci_lower']) and np.isfinite(record['ci_upper'])
            else f"{record['value_mean']:.3f}"
        ),
        axis=1,
    )
    data.loc[
        np.isclose(
            data['value_mean'],
            data.set_index(['landmark_h', 'horizon_h', 'target']).index.map(
                best_values
            ),
        ),
        'cell',
    ] = data.loc[
        np.isclose(
            data['value_mean'],
            data.set_index(['landmark_h', 'horizon_h', 'target']).index.map(
                best_values
            ),
        ),
        'cell',
    ].map(lambda cell: r'\textbf{' + cell + '}')
    canonical = data.pivot(
        index=['landmark_h', 'horizon_h', 'target'],
        columns='model_id',
        values='cell',
    ).reindex(columns=model_keys['model_id'])
    model_ids = model_keys['model_id'].tolist()
    lines = [
        r'\begin{table*}[t]',
        r'\centering',
        r'\caption{Test-set AUROC for acidemic delivery within a 1-hour forecast horizon.',
        r'Horizons of 12 and 48 hours are omitted.',
        r'The best value and 95\% confidence interval in each landmark--pH-category row are bolded.}',
        r'\label{tab:primary-auroc}',
        r'\scriptsize',
        r'\resizebox{\textwidth}{!}{%',
        r'\begin{tabular}{ll' + 'r' * len(model_ids) + r'}',
        r'\toprule',
        r'Landmark (h) & pH category & ' + ' & '.join(_latex_escape(family) for family in model_keys['family_label']) + r' \\',
        r'\midrule',
    ]
    for (landmark_h, horizon_h), landmark_table in canonical.groupby(
        level=['landmark_h', 'horizon_h'], sort=True
    ):
        landmark_table = landmark_table.droplevel(['landmark_h', 'horizon_h'])
        for row_index, target in enumerate(outcomes):
            rendered = landmark_table.loc[target].reindex(model_ids).fillna('--').tolist()
            outcome = (
                'Macro pH'
                if target == 'macro_pH'
                else f'pH $<$ {threshold_value(target):g}'
            )
            landmark_cell = (
                rf'\multirow{{{len(outcomes)}}}{{*}}{{{landmark_h:g}}}'
                if row_index == 0
                else ''
            )
            lines.append(
                landmark_cell
                + f' & {outcome} & '
                + ' & '.join(rendered)
                + r' \\'
            )
        lines.append(r'\addlinespace')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'}', r'\end{table*}', ''])
    table_path = TABLE_DIR / 'acidemic_delivery_horizon_auroc.tex'
    table_path.write_text('\n'.join(lines))
    print(f'Wrote {table_path}')


write_primary_auroc_table()


In [ ]:
def write_primary_ap_lift_table():
    required_columns = {
        "ap_lift",
        "ap_lift_ci_lower",
        "ap_lift_ci_upper",
    }
    missing_columns = required_columns.difference(metrics.columns)
    if missing_columns:
        raise KeyError(
            "AP-lift metrics are unavailable: "
            + ", ".join(sorted(missing_columns))
        )

    data = metrics.loc[
        metrics["split"].eq(SPLIT)
        & metrics["metric"].eq("auprc_acidemic_delivery_by_horizon")
        # & ~np.isclose(metrics["prediction_horizon_seconds"], 12 * 3600)
        # & ~np.isclose(metrics["prediction_horizon_seconds"], 48 * 3600)
        & np.isclose(metrics["prediction_horizon_seconds"], 1 * 3600)
    ].copy()
    if data.empty:
        print("No eligible AP-lift rows are available for the LaTeX table.")
        return

    data["landmark_h"] = data["landmark_seconds_since_start"] / 3600
    data["horizon_h"] = data["prediction_horizon_seconds"] / 3600
    data = data.groupby(
        [
            "landmark_h",
            "horizon_h",
            "family_label",
            "comparison_value",
            "target",
        ],
        as_index=False,
        dropna=False,
    ).agg(
        ap_lift_mean=("ap_lift", "mean"),
        ci_lower=("ap_lift_ci_lower", "mean"),
        ci_upper=("ap_lift_ci_upper", "mean"),
    )

    outcomes = sorted(
        data["target"].dropna().unique(),
        key=lambda target: (target != "macro_pH", threshold_value(target)),
    )
    best_values = data.groupby(
        ["landmark_h", "horizon_h", "target"],
        dropna=False,
    )["ap_lift_mean"].max()

    lines = [
        r"\begin{table*}[t]",
        r"\centering",
        r"\caption{Test-set AP lift for acidemic delivery within a 1-hour forecast horizon.",
        r"AP lift is average precision divided by outcome prevalence; a value of 1 corresponds to prevalence-level performance.",
        r"Horizons of 12 and 48 hours are omitted.",
        r"The best value and 95\% confidence interval in each landmark--outcome column are bolded.}",
        r"\label{tab:primary-ap-lift}",
        r"\scriptsize",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{ll" + "r" * len(outcomes) + r"}",
        r"\toprule",
        r"Landmark (h) & Model family & "
        + " & ".join(
            "Macro pH"
            if target == "macro_pH"
            else f"pH $<$ {threshold_value(target):g}"
            for target in outcomes
        )
        + r" \\",
        r"\midrule",
    ]

    for landmark_h, landmark_rows in data.groupby(
        "landmark_h", sort=True, dropna=False
    ):
        model_rows = list(
            landmark_rows.groupby(
                ["family_label", "comparison_value"],
                sort=True,
                dropna=False,
            )
        )
        for row_index, ((family, _), rows) in enumerate(model_rows):
            rows = rows.set_index("target")
            horizon_h = rows["horizon_h"].iloc[0]
            rendered = []

            for target in outcomes:
                if target not in rows.index:
                    rendered.append("--")
                    continue

                record = rows.loc[target]
                value = record["ap_lift_mean"]
                lower = record["ci_lower"]
                upper = record["ci_upper"]
                best = best_values.loc[(landmark_h, horizon_h, target)]

                cell = (
                    f"{value:.3f} ({lower:.3f}--{upper:.3f})"
                    if np.isfinite(lower) and np.isfinite(upper)
                    else f"{value:.3f}"
                )
                rendered.append(
                    r"\textbf{" + cell + "}"
                    if np.isclose(value, best)
                    else cell
                )

            landmark_cell = (
                rf"\multirow{{{len(model_rows)}}}{{*}}{{{landmark_h:g}}}"
                if row_index == 0
                else ""
            )
            lines.append(
                landmark_cell
                + f" & {_latex_escape(family)} & "
                + " & ".join(rendered)
                + r" \\"
            )
        lines.append(r"\addlinespace")

    lines.extend(
        [r"\bottomrule", r"\end{tabular}", r"}", r"\end{table*}", ""]
    )
    table_path = TABLE_DIR / "acidemic_delivery_horizon_ap_lift.tex"
    table_path.write_text("\n".join(lines))
    print(f"Wrote {table_path}")


write_primary_ap_lift_table()

In [ ]:
end_macro = metric_summary.loc[
    metric_summary["split"].eq("Test")
    & metric_summary["target"].eq("macro_pH")
    & metric_summary["metric"].eq("auroc_end_landmark")
    & metric_summary["minutes_before_delivery"].eq(0)
].copy()

display(
    end_macro[
        [
            "family_label",
            "comparison_value",
            "value_mean",
            "mean_run_ci_lower",
            "mean_run_ci_upper",
            "completed_runs",
        ]
    ].sort_values(["family_label", "comparison_value"])
)